In [1]:
# ==========================================
# 1. 必要なライブラリのインストール
# ==========================================
!pip install langchain-chroma langchain-ollama langchain-core langchain-community -q

# ==========================================d
# 2. Google Colab上でOllamaを確実にインストール＆起動
# ==========================================
import subprocess
import time
import os
import socket
import tempfile # For temporary log files

print("Ollamaをインストール中...")

# Ollamaインストールパスを設定 (スクリプトが/usr/localにインストールするため、これを反映)
ollama_install_dir = "/usr/local/bin"
ollama_executable_path = f"{ollama_install_dir}/ollama"

# 依存関係zstdをインストール（Ollamaの依存関係）
print("依存関係zstdをインストール中...")
subprocess.run(
    ["sudo", "apt-get", "update", "-qq"], # Update package list silently
    check=True,
    capture_output=True
)
subprocess.run(
    ["sudo", "apt-get", "install", "-y", "zstd"],
    check=True,
    capture_output=True
)
print("zstdのインストールに成功しました。")

# Ollamaインストールスクリプトをダウンロード
install_script_path = "/tmp/install_ollama.sh"
print(f"Ollamaインストールスクリプトをダウンロード中: https://ollama.com/install.sh")
subprocess.run(
    ["curl", "-fsSL", "https://ollama.com/install.sh", "-o", install_script_path],
    check=True,
    capture_output=True
)
print(f"Ollamaインストールスクリプトを {install_script_path} にダウンロードしました。")

# Ollamaインストールスクリプトを実行
print("Ollamaインストールスクリプトを実行中... (これには数分かかる場合があります)")
# OLLAMA_INSTALL_DIR を指定しても、スクリプトが /usr/local にインストールする場合があるため、
# Python側ではインストール後のパスを /usr/local/bin と仮定する。
# スクリプトにOLLAMA_INSTALL_DIRを渡しても無視される可能性があるため、ここでは明示的に渡さない。
install_process = subprocess.run(
    ["bash", install_script_path],
    check=True,
    capture_output=True
    # env=install_env # スクリプトが無視するため、ここでは渡さない
)
print(f"インストールスクリプトの標準出力:\n{install_process.stdout.decode()}")
if install_process.stderr:
    print(f"インストールスクリプトの標準エラー:\n{install_process.stderr.decode()}")

# PATHを更新してollamaコマンドが実行可能になるようにする
# 現在のPythonプロセスと将来のsubprocessのためにPATHを更新
if ollama_install_dir not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{ollama_install_dir}{os.pathsep}{os.environ.get('PATH', '')}"
print(f"Ollamaのインストールが完了しました。実行パス: {ollama_executable_path}")

# Ollamaサーバーをバックグラウンドで起動
print("Ollamaサーバーを起動中...")
if not os.path.exists(ollama_executable_path):
    raise FileNotFoundError(f"Ollama実行ファイルが見つかりません: {ollama_executable_path}. インストールに失敗した可能性があります。")

server_env = os.environ.copy() # Use the updated os.environ

# Redirect stdout and stderr to temporary files for debugging
stdout_log = tempfile.TemporaryFile(mode='w+')
stderr_log = tempfile.TemporaryFile(mode='w+')

try:
    server_process = subprocess.Popen(
        [ollama_executable_path, "serve"],
        stdout=stdout_log, # Redirect stdout to log file
        stderr=stderr_log, # Redirect stderr to log file
        env=server_env
    )

    # Ollamaサーバーが起動するまで待機する
    ollama_api_host = "127.0.0.1"
    ollama_api_port = 11434
    max_retries = 30
    for i in range(max_retries):
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.settimeout(1) # 1 second timeout
            s.connect((ollama_api_host, ollama_api_port))
            s.close()
            print("Ollamaサーバーが起動しました。")
            break
        except (socket.timeout, ConnectionRefusedError):
            print(f"Ollamaサーバーの起動を待機中... ({i+1}/{max_retries})")
            time.sleep(2) # Wait 2 seconds between retries
    else:
        # If the loop completes without breaking (server didn't start)
        print("-------------------------------------------------------------------")
        print("Ollamaサーバーが指定時間内に起動しませんでした。以下にログを表示します。")
        # Check if the server process has terminated and retrieve logs
        if server_process.poll() is not None:
            print("Ollamaサーバーが予期せず終了しました。ログを確認してください。")
            stdout_log.seek(0)
            stderr_log.seek(0)
            stdout_content = stdout_log.read()
            stderr_content = stderr_log.read()
            if stdout_content: print(f"Ollamaサーバーの標準出力:\n{stdout_content}")
            if stderr_content: print(f"Ollamaサーバーの標準エラー:\n{stderr_content}")
        else:
            print("Ollamaサーバーはまだ実行中ですが、ポートに応答していません。")
        print("-------------------------------------------------------------------")
        raise RuntimeError("Ollamaサーバーが指定時間内に起動しませんでした。")
finally:
    # Close temporary files
    stdout_log.close()
    stderr_log.close()

# 正しいOllamaのモデル名（チャット用）
chat_model_name = "lucas2024/llama-3-elyza-jp-8b:q5_k_m"
print(f"Ollamaから {chat_model_name} をダウンロード中... (約1〜2分)")

# モデルのプルが完了するまでしっかり待つ
pull_process = subprocess.run(
    [ollama_executable_path, "pull", chat_model_name],
    check=True, # Raise an exception if the command returns a non-zero exit code
    capture_output=True,
    env=server_env # Ensure updated PATH is used for `ollama pull`
)
print(f"モデルダウンロードの標準出力:\n{pull_process.stdout.decode()}")
if pull_process.stderr:
    print(f"モデルダウンロードの標準エラー:\n{pull_process.stderr.decode()}")

# 埋め込み用のモデル名
embedding_model_name = "nomic-embed-text"
print(f"Ollamaから埋め込み用モデル {embedding_model_name} をダウンロード中...")
pull_embedding_process = subprocess.run(
    [ollama_executable_path, "pull", embedding_model_name],
    check=True,
    capture_output=True,
    env=server_env
)
print(f"埋め込みモデルダウンロードの標準出力:\n{pull_embedding_process.stdout.decode()}")
if pull_embedding_process.stderr:
    print(f"埋め込みモデルダウンロードの標準エラー:\n{pull_embedding_process.stderr.decode()}")

print("OllamaとELYZAおよび埋め込みモデルの準備が完了しました！\n" + "="*40 + "\n")

# ==========================================
# 3. LangChain RAGパイプラインの構築と実行
# ==========================================
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Moved embedding_model and vector_store out of the function to initialize once
embedding_model = OllamaEmbeddings(model=embedding_model_name)

# Sample data definition moved here to be accessible for re-initialization if needed
sample_data = [
    Document(
        page_content="プロジェクト・アルファの内部コードネームは『シルバースァーファー』で、2026年10月にリリース予定です。",
        metadata={"source": "project_alpha.txt"}
    ),
    Document(
        page_content="プロジェクト・アルファの予算は120万ドルで、すべてマーケティングチームが管理しています。",
        metadata={"source": "finance_2026.txt"}
    ),
    Document(
        page_content="プロジェクト・アルファの新しいチームメンバーは佐藤、田中、鈴木です。",
        metadata={"source": "team_update.txt"}
    ),
    Document(
        page_content="プロジェクト・アルファの会議は毎週火曜日の午前10時に開催されます。",
        metadata={"source": "meeting_schedule.txt"}
    ),
    Document(
        page_content="プロジェクト・ベータは2027年にリリース予定の別のプロジェクトです。",
        metadata={"source": "project_beta.txt"} # Slightly less relevant distractor
    ),
    Document(
        page_content="プロジェクト・アルファの主な目標は市場シェアを20%拡大することです。",
        metadata={"source": "strategic_plan.txt"}
    )
]

# Initialize vector store once with all documents
vector_store = Chroma.from_documents(documents=sample_data, embedding=embedding_model)

def run_colab_rag(user_query: str, k: int):
    print(f"\n--- RAG実行: k={k} ---")
    retriever = vector_store.as_retriever(search_kwargs={"k": k})

    # まず、実際にどのドキュメントが取得されたかを確認する
    retrieved_docs = retriever.invoke(user_query)
    print(f"\n--- 取得されたドキュメント (k={k}) ---")
    for i, doc in enumerate(retrieved_docs):
        print(f"ドキュメント {i+1} (ソース: {doc.metadata.get('source', '不明')}):\n{doc.page_content}\n")

    # プロンプトテンプレート
    prompt_template = """
    あなたは誠実で優秀な日本人のアシスタントです。
    必ず、以下の【提供された文脈】のみに基づいて質問に答えてください。
    文脈から判断できない場合は、「ドキュメント内に答えが見つかりませんでした」と答えてください。

    【提供された文脈】:
    {context}

    【質問】:
    {question}

    【回答】:
    """
    prompt = ChatPromptTemplate.from_template(prompt_template)
    llm = ChatOllama(model=chat_model_name, temperature=0) # Use generative chat model

    # チェーンの結合
    # ここでretrieverの代わりにretrieved_docsをcontextとして渡す
    rag_chain = (
        {"context": lambda x: retrieved_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 実行
    print(f"ユーザーの質問: {user_query}")
    print("GPUを活用して高速回答を生成中...")

    response = rag_chain.invoke(user_query)
    print(f"\n【ELYZAからの回答 (k={k})】:\n{response}")
    return response, retrieved_docs

# The initial `run_colab_rag()` call is removed as it will be demonstrated in new cells.


Ollamaをインストール中...
依存関係zstdをインストール中...
zstdのインストールに成功しました。
Ollamaインストールスクリプトをダウンロード中: https://ollama.com/install.sh
Ollamaインストールスクリプトを /tmp/install_ollama.sh にダウンロードしました。
Ollamaインストールスクリプトを実行中... (これには数分かかる場合があります)
インストールスクリプトの標準出力:

インストールスクリプトの標準エラー:
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Ollamaのインストールが完了しました。実行パス: /usr/local/bin/ollama
Ollamaサーバーを起動中...
Ollamaサーバーの起動を待機中... (1/30)
Ollamaサーバーが起動しました。
Ollamaから lucas2024/llama-3-elyza-jp-8b:q5_k_m をダウンロード中... (約1〜2分)
モデルダウンロードの標準出力:

モデルダウンロードの標準エラー:
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling 

### 4. PDFドキュメントの読み込みとベクトルストアの更新

アップロードされたPDFドキュメントを読み込み、チャンクに分割し、それをベクトルストアに追加します。これにより、RAGパイプラインがPDFの内容に基づいて質問に答えられるようになります。

In [2]:
!pip install langchain-community -q
print("langchain-communityをインストールしました。")

langchain-communityをインストールしました。


In [3]:
print('pypdfをインストール中...')
!pip install pypdf -q
print('pypdfのインストールが完了しました。')

pypdfをインストール中...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 16.9 MB/s eta 0:00:00
pypdfのインストールが完了しました。


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf"
print(f"PDFドキュメントをロード中: {pdf_path}")
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"ロードされたドキュメントのページ数: {len(documents)}")

# ドキュメントをチャンクに分割
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"分割されたチャンクの数: {len(chunks)}")

# 新しいベクトルストアをPDFドキュメントのチャンクで初期化
# 既存の `embedding_model` を使用
print("PDFチャンクを使用して新しいベクトルストアを作成中...")
vector_store_pdf = Chroma.from_documents(documents=chunks, embedding=embedding_model)
print("PDFチャンクを含むベクトルストアが正常に作成されました。")

# RAGチェーンの `vector_store` を更新してPDFの内容を使用できるようにする
# ここでは、元の `vector_store` を上書きしてPDFデータを使用するようにします。
# 必要に応じて、両方を組み合わせて使用することも可能です。
vector_store = vector_store_pdf
print("RAGパイプラインがPDFデータを使用するように設定されました。")

/tmp/ipykernel_5281/3256270761.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDFドキュメントをロード中: /content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf
ロードされたドキュメントのページ数: 6
分割されたチャンクの数: 13
PDFチャンクを使用して新しいベクトルストアを作成中...
PDFチャンクを含むベクトルストアが正常に作成されました。
RAGパイプラインがPDFデータを使用するように設定されました。


### PyPDFLoaderによるPDF読み込みとメタデータの確認

`PyPDFLoader`はPDFをページごとに読み込み、各ドキュメント（ページ）に自動的にファイルパス（`source`）やページ番号（`page`）などのメタデータを付与します。チャンクに分割された後も、このメタデータは各チャンクに引き継がれます。

In [12]:
# 既存のpdf_pathとembedding_modelを使用します。
# PDFの再ロード
loader = PyPDFLoader(pdf_path)
documents_with_metadata_example = loader.load()
print(f"ロードされたドキュメントのページ数: {len(documents_with_metadata_example)}")

# ドキュメントをチャンクに分割
text_splitter_example = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks_with_metadata_example = text_splitter_example.split_documents(documents_with_metadata_example)
print(f"分割されたチャンクの数: {len(chunks_with_metadata_example)}")

# メタデータを確認するために、最初の数個のチャンクの内容とメタデータを表示します。
print("\n=== 最初の3つのチャンクの内容とメタデータ ===")
for i, chunk in enumerate(chunks_with_metadata_example[:3]):
    print(f"\n--- チャンク {i+1} ---")
    print(f"Content (抜粋): {chunk.page_content[:200]}...") # 内容の一部を表示
    print(f"Metadata: {chunk.metadata}")

# このチャンクをChromaに格納することも、以前のセルで行ったように簡単です。
# vector_store_pdf_example = Chroma.from_documents(
#     documents=chunks_with_metadata_example,
#     embedding=embedding_model
# )
# print("\nチャンクがChromaに正常に格納されました。(コメントアウトされています)")

ロードされたドキュメントのページ数: 6
分割されたチャンクの数: 13

=== 最初の3つのチャンクの内容とメタデータ ===

--- チャンク 1 ---
Content (抜粋): [PR]ボッシュ×⽇⽴ルームエアコン「⽩くまくん」のシナジーと...
[ ログイン新規ID登録] ご利⽤ガイド閲覧履歴
ペンタックス PENTAX K-1 Mark II ボディ 価格⽐較
ホームカメラ    PENTAX K-1 Mark II ボディデジタル⼀眼カメラペンタックス
※レンズは別売です
メーカー希望⼩売価格︓オープン2018年4⽉20⽇ 発売
タイプ ⼀眼レフ 画素数 3677万...
Metadata: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-16T06:33:37+00:00', 'title': 'ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com', 'moddate': '2026-06-16T06:33:37+00:00', 'source': '/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

--- チャンク 2 ---
Content (抜粋): 1,756円相当P
無料 問合せ★★楽天市場★★なら、いつでもポイントがもらえる、使える︕
楽天市場楽天ビックカメラ ショップに問い合わせる
送料込み実質価格(商品価格＋送料ーポイント)
価格（差額） 送料 在庫発送⽬安 ショップ情報
(最安)
191,470円 無料 問合せ■ショッピングクレジットのご利⽤で分割48回まで⾦利無料︕
カメラのキタムラ ショップに問い合わせる
(+21,280)
2...
Metadata: {'producer': 'Skia/PDF m149', '

### 5. PDFドキュメントに対するRAGの実行

PDFドキュメントから情報を取得できるか確認するために、PDFの内容に関する質問で `run_colab_rag` 関数を実行します。

In [10]:
# PDFの内容に関する質問
query_pdf = "ペンタックス K-1 Mark IIの有効画素数と主な特徴を教えてください。"
response_pdf, retrieved_docs_pdf = run_colab_rag(user_query=query_pdf, k=3)

print(f"\n【PDFに対するELYZAからの回答】:\n{response_pdf}")


--- RAG実行: k=3 ---

--- 取得されたドキュメント (k=3) ---
ドキュメント 1 (ソース: /content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf):
[PR]ボッシュ×⽇⽴ルームエアコン「⽩くまくん」のシナジーと...
[ ログイン新規ID登録] ご利⽤ガイド閲覧履歴
ペンタックス PENTAX K-1 Mark II ボディ 価格⽐較
ホームカメラ    PENTAX K-1 Mark II ボディデジタル⼀眼カメラペンタックス
※レンズは別売です
メーカー希望⼩売価格︓オープン2018年4⽉20⽇ 発売
タイプ ⼀眼レフ 画素数 3677万画素(総画素)3640万画素(有効画素)
撮像素⼦ フルサイズ35.9mm×24mmCMOS 重量 925 g
メーカー公式情報メーカートップページ メーカー製品情報ページ メーカー仕様表プレスリリース
ペンタックスKシリーズ
ペンタックス
PENTAX K-1 Mark II ボディ
最安価格
191,470円
中古最安価格
131,000円 価格推移グラフ
売れ筋ランキング
(1274製品中)47位
レビュー
(38件)4.69
クチコミ
2664件
お気に⼊り
(711⼈)お気に⼊り
スペック・仕様 すべてのスペック・仕様を⾒る
★★楽天市場★★なら、いつでもポイントがもらえる、使える︕
PENTAX K-1 Mark II ボディ
価格︓193,232円
1,756円相当P
絞り込み
サービス
⽀払⽅法
在庫・発送の⽬安
ショッピングモール別価格
量販店別価格
最安値ショップ 送料込み実質価格(商品価格＋送料ーポイント)
全14ショップで最安
価格（差額）送料 在庫発送⽬安 ショップ情報
(最安)
191,470円 無料 問合せ■ショッピングクレジットのご利⽤で分割48回まで⾦利無料︕
カメラのキタムラ
クレカ振込その他
ショップに問い合わせる
送料込み実質価格(商品価格＋送料ーポイント)
価格（差額） 送料 在庫発送⽬安 ショップ情報
(+1,762)
193,232円
1,756円相当P
無料 問合せ★★楽天市場★★なら、いつでもポイントがもらえる、使える︕
楽天市場楽天ビックカメラ ショップに問い合わせる
送料込み実質価格

## Chromaの永続化 (Persistence) について

Chromaはデフォルトではインメモリで動作しますが、データをディスクに保存して再利用できるように**永続化**することができます。永続化を設定しない場合、ランタイムが終了するとデータは失われます。

### 永続化の重要性

*   **データ保持:** ノートブックのセッション終了後もベクトルストアのデータを保持します。
*   **効率性:** 毎回ドキュメントを再インデックスする手間と時間を省きます。
*   **スケーラビリティ:** 大規模なデータセットを扱う際に不可欠です。

### 永続化の設定方法

`Chroma.from_documents()` または `Chroma()` を初期化する際に、`persist_directory` パラメータでデータを保存するディレクトリを指定します。**`persist_directory`が指定された場合、ドキュメントの追加時にデータは自動的にディスクに保存されるため、別途 `vector_store.persist()` を呼び出す必要はありません。**

### 例:


In [8]:
!pip install langchain-chroma langchain-ollama -q
import os
import shutil
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings # Import OllamaEmbeddings

# 永続化するディレクトリを指定
persist_directory = "/tmp/chroma_db_persistent" # ディレクトリを/tmpに変更

# 既に存在する場合は削除（実験用）
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
    print(f"既存のディレクトリ '{persist_directory}' を削除しました。")

# サンプルドキュメント
sample_docs_persistent = [
    Document(page_content="Chromaは永続化をサポートしています。", metadata={"source": "chroma_doc"}),
    Document(page_content="persist_directoryを指定することでデータを保存できます。", metadata={"source": "chroma_doc"})
]

# embedding_model をこのセル内で再定義
embedding_model_name = "nomic-embed-text" # 元のセルで定義されたモデル名を使用
embedding_model = OllamaEmbeddings(model=embedding_model_name)

# 永続化するChromaベクトルストアを作成
print("永続化Chromaストアを作成中...")
vector_store_persistent = Chroma.from_documents(
    documents=sample_docs_persistent,
    embedding=embedding_model, # 既存の埋め込みモデルを使用
    persist_directory=persist_directory
)

# 変更をディスクに書き込む (この行は不要です。persist_directory指定時に自動保存されます)
# vector_store_persistent.persist()
print(f"Chromaベクトルストアが '{persist_directory}' に永続化されました。")

# 後でこのストアをロードするには:
# from langchain_chroma import Chroma
# from langchain_ollama import OllamaEmbeddings
# embedding_model_name = "nomic-embed-text" # ロード時に同じembedding_modelを再初期化
# embedding_model = OllamaEmbeddings(model=embedding_model_name)
# loaded_vector_store = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)
# print("永続化Chromaストアが正常にロードされました。")

query_persistent = "Chromaの機能について教えてください。"
retrieved_docs_persistent = vector_store_persistent.as_retriever(search_kwargs={"k": 1}).invoke(query_persistent)

print(f"\n【永続化ストアからの検索結果】:")
for i, doc in enumerate(retrieved_docs_persistent):
    print(f"ドキュメント {i+1}:")
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")


永続化Chromaストアを作成中...
Chromaベクトルストアが '/tmp/chroma_db_persistent' に永続化されました。

【永続化ストアからの検索結果】:
ドキュメント 1:
  Content: Chromaは永続化をサポートしています。
  Metadata: {'source': 'chroma_doc'}


### 6. Chromaへのメタデータ追加の例

ここでは、`Document`オブジェクトにカスタムメタデータを追加し、そのメタデータを含むChromaベクトルストアを作成する例を示します。これにより、RAGパイプラインでより高度なフィルタリングや情報管理が可能になります。

In [9]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# メタデータを含むサンプルドキュメントを作成
# 各Documentのmetadata辞書に任意のキーと値を設定できます。
documents_with_metadata = [
    Document(
        page_content="プロジェクトAの予算は2024年に100万ドルと承認されました。",
        metadata={
            "source_id": "proj-A-budget-2024",
            "version": "1.0",
            "updated_at": "2024-01-15",
            "access_role": "finance",
            "department": "Finance"
        }
    ),
    Document(
        page_content="プロジェクトBの最終報告書は2023年12月に提出されました。",
        metadata={
            "source_id": "proj-B-report-2023",
            "version": "final",
            "updated_at": "2023-12-20",
            "access_role": "public",
            "department": "R&D"
        }
    ),
    Document(
        page_content="新入社員向けのオンボーディングプロセスに関するガイドライン。",
        metadata={
            "source_id": "hr-onboarding-guide",
            "version": "3.1",
            "updated_at": "2024-03-01",
            "access_role": "hr",
            "department": "HR"
        }
    )
]

print("メタデータ付きドキュメント:\n")
for doc in documents_with_metadata:
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

# メタデータ付きドキュメントで新しいChromaベクトルストアを初期化
# embedding_modelは既に定義されているものを使用します。
print("メタデータ付きドキュメントを使用してChromaベクトルストアを作成中...")
vector_store_with_metadata = Chroma.from_documents(
    documents=documents_with_metadata,
    embedding=embedding_model
)
print("メタデータ付きのChromaベクトルストアが作成されました。")

# メタデータを使って検索する例（ここではフィルタリングは行いませんが、
# as_retriever(search_kwargs={"k": k, "where": {"access_role": "finance"}})のように使用できます）
query_metadata = "プロジェクトAの予算はいくらですか？"
retriever_metadata = vector_store_with_metadata.as_retriever(search_kwargs={"k": 1})
retrieved_docs_metadata = retriever_metadata.invoke(query_metadata)

print(f"\n【メタデータ付きChromaからの検索結果】:")
for i, doc in enumerate(retrieved_docs_metadata):
    print(f"ドキュメント {i+1}:")
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")


メタデータ付きドキュメント:

Content: プロジェクトAの予算は2024年に100万ドルと承認されました。
Metadata: {'source_id': 'proj-A-budget-2024', 'version': '1.0', 'updated_at': '2024-01-15', 'access_role': 'finance', 'department': 'Finance'}

Content: プロジェクトBの最終報告書は2023年12月に提出されました。
Metadata: {'source_id': 'proj-B-report-2023', 'version': 'final', 'updated_at': '2023-12-20', 'access_role': 'public', 'department': 'R&D'}

Content: 新入社員向けのオンボーディングプロセスに関するガイドライン。
Metadata: {'source_id': 'hr-onboarding-guide', 'version': '3.1', 'updated_at': '2024-03-01', 'access_role': 'hr', 'department': 'HR'}

メタデータ付きドキュメントを使用してChromaベクトルストアを作成中...
メタデータ付きのChromaベクトルストアが作成されました。

【メタデータ付きChromaからの検索結果】:
ドキュメント 1:
  Content: プロジェクト・アルファの主な目標は市場シェアを20%拡大することです。
  Metadata: {'source': 'strategic_plan.txt'}


## RAG性能と`k`の値の関連性のデモンストレーション (Demonstrating RAG Performance vs. `k` Value)

ここでは、レトリバーが取得するドキュメントの数（`k`）が、RAGモデルの回答品質にどのように影響するかをデモンストレーションします。

まず、最も関連性の高いドキュメントのみを取得する小さな`k`（例: `k=2`）で質問し、その後に全てのドキュメントを取得する大きな`k`（例: `k=6`）で同じ質問をします。

**質問:** `プロジェクト・アルファのコードネームとリリース時期を教えて。`

In [ ]:
# ケース1: 小さなkの値 (k=2) でRAGを実行
# この設定では、レトリバーは上位2つの最も関連性の高いドキュメントのみを取得します。
query = "プロジェクト・アルファのコードネームとリリース時期を教えて。"
response_k2, retrieved_docs_k2 = run_colab_rag(user_query=query, k=2)

In [ ]:
# ケース2: 大きなkの値 (k=6, 全ドキュメント) でRAGを実行
# この設定では、レトリバーは全ての6つのドキュメントを取得します。
# 無関係な情報が混ざることで、回答が曖昧になる可能性があります。
query = "プロジェクト・アルファのコードネームとリリース時期を教えて。"
response_k6, retrieved_docs_k6 = run_colab_rag(user_query=query, k=6)

## 結果の考察 (Analysis of Results)

上記の2つのケースを比較してみましょう。今回は、レトリバーが実際にどのようなドキュメントを取得したかも確認できます。

**`k=2` の場合:**
*   **取得されたドキュメント:** (上記出力参照)
*   **LLMの回答:** モデルは質問に関連性の高い情報のみを受け取ったため、簡潔で正確な回答を生成する可能性が高いですが、もし必要な情報が取得された2つのドキュメントに含まれていなければ「ドキュメント内に答えが見つかりませんでした」と回答します。今回の例では、`k=2`で重要な情報を含むドキュメントが選ばれなかったため、正しい回答が得られませんでした。

**`k=6` の場合 (全てのドキュメント):**
*   **取得されたドキュメント:** (上記出力参照)
*   **LLMの回答:** レトリバーがより多くのドキュメント（関連性の低いものを含む）を取得しました。RAGシステムでは、モデルが一度に処理するコンテキストの量が増えると、以下の問題が発生する可能性があります。
    1.  **情報の希釈 (Information Dilution):** 関連性の低い情報がノイズとなり、モデルが本当に重要な情報を特定しにくくなる。
    2.  **幻覚 (Hallucination) の増加:** モデルが文脈全体を「理解」しようとする際に、誤った関連性を見つけたり、事実と異なる情報を生成したりする可能性が高まる。
    3.  **応答速度の低下:** 処理するトークン数が増えるため、モデルの推論時間が長くなる。
    4.  **「真ん中」の問題 (Lost in the Middle):** 重要な情報が長いコンテキストの真ん中にある場合、モデルがそれを見落とす傾向がある。

このデモンストレーションでは、`k`の値がRAGの性能にどのように影響するか、特に情報の過多が回答品質を低下させる可能性があることを示しています。
しかし、`k`が小さすぎると、必要な情報がそもそもコンテキストに含まれない可能性もあります。

結論として、RAGの性能を最適化するためには、**質問に関連性の高いドキュメントを効率的に取得し、かつその数を適切に制限する**ことが重要です。適切な`k`の値を見つけることは、リトリーバーの精度とモデルの処理能力のバランスを取る作業となります。

## メモ: RAGにおける`k`とSQLの`SELECT TOP N`について

RAG (Retrieval-Augmented Generation) における`k`は、SQLの`SELECT TOP N`に非常に似ています。

**共通点:**
*   どちらも、ある基準に基づいて上位N件（この場合は`k`件）の結果を取得するという点です。

**RAGにおける`k`:**
*   `k`は、レトリバー（情報検索コンポーネント）がベクトルデータベースから取得するドキュメントの数を指定します。
*   レトリバーは、ユーザーのクエリとデータベース内の各ドキュメントを埋め込みモデルでベクトル化し、それらの**ベクトル類似度**（コサイン類似度など）を計算します。そして、類似度スコアが最も高い上位`k`個のドキュメントを選択し、それらをLLM（大規模言語モデル）にコンテキストとして渡します。
*   つまり、RAGの`k`は「最もクエリに近い（類似度の高い）`k`個のドキュメントを抽出する」と解釈できます。

**SQLの`SELECT TOP N`との違い:**
*   SQLの`SELECT TOP N`が主にテーブルのカラムの値による並べ替え（`ORDER BY`句）に基づいて行を抽出するのに対し、RAGの`k`は**意味的な類似度**に基づいてドキュメントを抽出します。

**`k`の重要性:**
*   `k`が小さすぎると、回答に必要な情報がそもそもLLMに提供されないリスクがあります。
*   `k`が大きすぎると、関連性の低い情報（ノイズ）が混入し、「情報の希釈」や「Lost in the Middle（真ん中の情報が見落とされる）」といった問題が発生し、LLMの回答品質が低下する可能性があります。
*   適切な`k`の値を見つけることは、RAGシステムの性能を最適化する上で非常に重要です。

## Resources: Chromaにおけるレコードの検出と更新

Chromaのようなベクトルストアに格納されたレコードを管理する際、特にPDFファイルから生成されたデータの場合、元のPDFファイル名（`source`メタデータ）は非常に重要な識別子となります。これにより、特定のPDFに由来するチャンクを検出したり、更新・削除したりすることが可能になります。

### 1. PDFファイル名に基づいてレコードを検出する方法

`PyPDFLoader`で読み込まれたドキュメントが`RecursiveCharacterTextSplitter`でチャンクに分割されると、各チャンクの`metadata`には元のPDFのファイルパスが`source`として含まれます。この`source`メタデータを利用して、特定のPDFに属するチャンクをクエリで検索できます。

In [13]:
# 検出したいPDFのファイルパス
# 既存のpdf_path変数を使用します。
# pdf_path = "/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf"

# vector_store_pdf_exampleは前のセルで作成されていますが、ここでは現在のvector_storeを使用します。
# 現在のvector_storeは、最後にPDFデータで上書きされたものです。

# 検索したいPDFのソースメタデータ
target_source = pdf_path # /content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf

print(f"ソースが '{target_source}' のドキュメントを検索中...")

# vector_storeから直接フィルタリングしてドキュメントを取得
# Chromaの.get()メソッドはIDで取得しますが、メタデータによるフィルタリングは.query()またはretrieverのwhere句で行います。
# まずはretrieverを使って、where句でフィルタリングする方法を検討します。

# メタデータフィルタリングをサポートするretrieverを作成
# 注意: Chromaのretrieverでのwhere句フィルタリングは、Chromaオブジェクトそのものに対して行うのではなく、
# as_retriever()のsearch_kwargsとして渡す形になります。
retriever_filtered = vector_store.as_retriever(search_kwargs={"filter": {"source": target_source}})

# 任意のクエリでretrieverを呼び出すことで、フィルタリングされたドキュメントを取得できます。
# ここではダミークエリを使用します。
dummy_query = ""
filtered_docs = retriever_filtered.invoke(dummy_query)

if filtered_docs:
    print(f"'{target_source}' に関連する {len(filtered_docs)} 個のドキュメントが見つかりました:")
    for i, doc in enumerate(filtered_docs[:5]): # 最初の方のドキュメントをいくつか表示
        print(f"  --- ドキュメント {i+1} ---")
        print(f"  内容 (抜粋): {doc.page_content[:100]}...")
        print(f"  メタデータ: {doc.metadata}")
else:
    print(f"'{target_source}' に関連するドキュメントは見つかりませんでした。")

ソースが '/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf' のドキュメントを検索中...
'/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf' に関連する 4 個のドキュメントが見つかりました:
  --- ドキュメント 1 ---
  内容 (抜粋): リコー、フルサイズ⼀眼レフ「PENTAX K-1 Mark II」を4/20より発売
2018年3⽉23⽇
リコー、フルサイズデジタル⼀眼レフ「PENTAX K-1 Mark II」を4⽉下旬発売
2...
  メタデータ: {'total_pages': 6, 'moddate': '2026-06-16T06:33:37+00:00', 'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'source': '/content/ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com.pdf', 'creationdate': '2026-06-16T06:33:37+00:00', 'title': 'ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com', 'page': 4, 'page_label': '5'}
  --- ドキュメント 2 ---
  内容 (抜粋): 1,756円相当P
無料 問合せ★★楽天市場★★なら、いつでもポイントがもらえる、使える︕
楽天市場楽天ビックカメラ ショップに問い合わせる
送料込み実質価格(商品価格＋送料ーポイント)
価格（差額）...
  メタデータ: {'page': 0, 'page_label': '1', 'title': 'ペンタックス PENTAX K-1 Mark II ボディ 価格比較 - 価格.com', 'moddate': '2026-06-16T06:33:37+00:00', 'source'

### 2. PDFファイル名に基づいてレコードを更新/削除する方法

Chromaで特定のPDFファイルに由来するレコードを更新または削除するには、以下の手順が考えられます。

#### レコードの削除
Chromaの`delete`メソッドを使用して、メタデータフィルタに基づいてドキュメントを削除できます。

```python
# 削除したいPDFのソースメタデータ
target_source_to_delete = pdf_path # 例

print(f"ソースが '{target_source_to_delete}' のドキュメントを削除中...")

# メタデータフィルタリングを使用してドキュメントを削除
# ids = vector_store.get(where={"source": target_source_to_delete})['ids'] # IDを事前に取得する必要があるかもしれません
# vector_store.delete(ids=ids)
# LangChainのChromaラッパーでは直接メタデータフィルタでの削除は提供されていません。
# 生のChromaクライアントを使うか、ChromaのIDで削除する必要があります。

# 一般的な方法として、IDを取得してから削除する例（概念コード）:
# docs_to_delete = retriever_filtered.invoke("") # 先ほどのようにフィルタリングしてドキュメントを取得
# ids_to_delete = [doc.metadata.get('id') for doc in docs_to_delete if doc.metadata.get('id')] # DocumentにChromaのIDが含まれている場合
# if ids_to_delete:
#     vector_store._collection.delete(ids=ids_to_delete)
#     print(f"{len(ids_to_delete)} 個のドキュメントが正常に削除されました。")
# else:
#     print("削除するドキュメントは見つかりませんでした。")

# より簡単なのは、Chromaを再初期化する際に`persist_directory`を削除することですが、これは全てを削除します。
# 厳密に特定のソースだけを削除したい場合は、より低レベルなChromaのAPIを直接使う必要があります。
# または、LangChainのChromaオブジェクトがサポートするdelete()メソッドにフィルタを渡すことを試みます。

# LangChain Chromaでは、DocumentオブジェクトのIDで削除するか、コレクション全体をクリアするかに限定されることが多いです。
# メタデータでの直接削除は、ChromaDBのPythonクライアントのCollectionオブジェクトのdelete()メソッドで行えます。
# from chromadb.utils import embedding_functions
# from chromadb import PersistentClient
# client = PersistentClient(path=persist_directory) # persist_directoryを仮定
# collection = client.get_or_create_collection(name="langchain", embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(model_name="nomic-embed-text"))
# collection.delete(where={"`source`": target_source_to_delete})
# print(f"ソースが '{target_source_to_delete}' のドキュメントがChromaコレクションから削除されました。（ChromaクライアントAPIを使用）")
```

#### レコードの更新（再インデックス）
通常、Chromaのようなベクトルストアでは、「更新」という操作は「既存のレコードを削除し、新しいレコードを追加する」という形で実装されます。これは、チャンクの内容が変更された場合、そのチャンクの埋め込みベクトルも変更されるため、再生成する必要があるからです。

**手順:**
1.  **既存レコードの検出と削除:** 上記の「レコードの削除」セクションで説明したように、古いPDFファイルに由来するチャンクを特定し、Chromaから削除します。
2.  **新しいドキュメントの読み込みとチャンク化:** 変更されたPDFファイルを`PyPDFLoader`で再度読み込み、`RecursiveCharacterTextSplitter`でチャンクに分割します。
3.  **新しいレコードの追加:** 新しいチャンクを`Chroma.from_documents()`または`vector_store.add_documents()`メソッドを使ってChromaに追加します。

このプロセスにより、RAGシステムは常に最新かつ正確な情報に基づいて応答できるようになります。特に、`persist_directory`を使用している場合は、変更をディスクに永続化させることが重要です。

## これまでのOllamaセットアップに関する課題とその解決策 (Issues and Solutions for Ollama Setup So Far)

このセクションでは、Ollamaのセットアップ中に直面した主な課題と、それらをどのように解決したかをまとめます。

### 1. `zstd` 依存関係の不足
*   **課題:** Ollamaのインストールスクリプトが、必要な圧縮ユーティリティ `zstd` がシステムにないために失敗しました。
*   **解決策:** `sudo apt-get install -y zstd` コマンドを実行して `zstd` を事前にインストールしました。

### 2. Ollamaインストールパスの不一致
*   **課題:** `OLLAMA_INSTALL_DIR` 環境変数を設定しても、Ollamaの公式インストールスクリプトが `/usr/local/bin` にOllamaをインストールするという独自のロジックを持っていました。これにより、Pythonスクリプトが誤ったパスのOllama実行ファイルを探索していました。
*   **解決策:** Pythonスクリプト内の `ollama_install_dir` および `ollama_executable_path` 変数を `/usr/local/bin` に明示的に設定し、スクリプトの動作に合わせて調整しました。

### 3. `--embeddings` フラグが認識されない
*   **課題:** 最初、LangChain `OllamaEmbeddings` から「This server does not support embeddings. Start it with --embeddings」というエラーメッセージを受け取ったため、`ollama serve` コマンドに `--embeddings` フラグを追加しようとしました。しかし、このフラグはOllamaの `serve` コマンドには存在しませんでした。
*   **解決策:** `ollama serve --help` の出力を確認し、`--embeddings` フラグが有効なオプションではないことを確認しました。`ollama serve` はフラグなしで実行し、モデルが埋め込み機能を提供することに依存するようにしました。

### 4. 埋め込み機能のモデル固有の要件
*   **課題:** `llama-3-elyza-jp-8b` モデルを `OllamaEmbeddings` に直接使用したところ、`ResponseError: This server does not support embeddings` というエラーが再度発生しました。これは、生成モデルが必ずしも埋め込み機能を提供しないことを示唆していました。
*   **解決策:** 生成タスクには `lucas2024/llama-3-elyza-jp-8b:q5_k_m` を引き続き使用しつつ、**専用の埋め込みモデル**である `nomic-embed-text` をOllamaから別途ダウンロードし、`OllamaEmbeddings` にそのモデルを使用するように変更しました。これにより、RAGパイプラインが正しく機能するようになりました。

これらの課題を一つずつ解決することで、Colab環境でOllamaを使用したRAGパイプラインを正常に構築することができました。

## RAGと`k`に関する追加メモ

RAG (Retrieval-Augmented Generation) は、自然言語処理において急速に進化し、活発に研究されている分野です。リトリーバルと生成を組み合わせるという核となるアイデアは以前から存在していましたが、大規模言語モデル (LLM) やベクトルデータベースの進歩に伴い、その実用的な応用と有効性が大幅に成熟しました。多くの研究者が、その性能と効率を向上させ、リトリーバル精度の課題（現在の埋め込みモデルで直面しているような問題）や、ノイズの多いまたは関連性のないリトリーブされた情報の処理などの問題に取り組んでいます。

したがって、RAGは現在も活発に研究開発が進められている分野ですが、同時に現実世界の様々なアプリケーションで広く採用されるほどに成熟しています。

## Memo

This memo records the discussion about RAG performance and `chunk_size`.

`chunk_size` is a crucial parameter in RAG:
*   **Too small:** Can lead to loss of context if information is split across chunks. May increase retrieval overhead.
*   **Too large:** Can cause information dilution (noise), the 'Lost in the Middle' problem, increased LLM processing cost/time, and may exceed the LLM's context window limits.

The optimal `chunk_size` depends on the document type, query type, and LLM's context window. Experimentation is key to finding the right balance.